# 音の情報処理5：音声認識

ここでは、OpenAI社が開発した音声認識エンジン Whisper をインストールして音声認識を試してみましょう。


## 1. 音声データの用意

以下を実行して、外部ファイルをダウンロードしてください。   
**このセルはColaboratoryを起動するたびに必要となります**

In [ ]:
import urllib.request
!mkdir fig sound
urllib.request.urlretrieve('https://park.itc.u-tokyo.ac.jp/yamakata-lab/lecture/mediaproc/mediaproc2/sound/speech.wav', './sound/speech.wav')

音を聞いてみましょう。   
これは女性の音声で、[Wikipediaの音声認識に関する記事(ref. 2023.05.22)](https://ja.wikipedia.org/wiki/%E9%9F%B3%E5%A3%B0%E8%AA%8D%E8%AD%98)の冒頭の文「音声認識は声がもつ情報をコンピュータに認識させるタスクの総称である。ヒトの音声認識と対比して自動音声認識とも呼ばれる」を読み上げたものです。

In [ ]:
import IPython
IPython.display.Audio('sound/speech.wav')

## 2. ライブラリのインストール

今回は、音声認識を実装するためのライブラリとして[openai](https://openai.com/)の音声認識ツールである[Whisper](https://github.com/openai/whisper)を使用します。   
まずはライブラリをインストールしましょう。

In [ ]:
!pip install -q openai-whisper

## 3. 音声認識の実行
モデルをダウンロードしたのち、1.でダウンロードした音声を入力して認識結果を出力してみましょう。  
ここはMultilingual modelの`base`を使用しています。

In [ ]:
import whisper

# モデルをダウンロード
model = whisper.load_model("base")

# 音声を入力し結果を得る
result = model.transcribe("sound/speech.wav")

# 認識結果を出力
print(result["text"])

## 4. モデルの選択

[whisperのmodel card](https://github.com/openai/whisper/blob/main/model-card.md)によれば、モデルは`tiny`から`large`まで(最新は`large-v2`)選択できるようです。  
上で使用した`base`はパラメータサイズが74 Mですが、`medium`は769 Mと、10倍以上の差があります。   
これらによって結果がどのように変わるか、以下のコードで確かめてみましょう。

In [ ]:
# モデルをダウンロード
model = whisper.load_model("medium")

# 音声を入力し結果を得る
result = model.transcribe("sound/speech.wav")

# 認識結果を出力
print(result["text"])

## 5. 音声認識の仕組み

Whisperはどのような仕組みで動いているのでしょうか？   
[Whisperのページ](https://github.com/openai/whisper)にその仕組みが図示されています。  
音声は"Log-Mel Spectrogram"に変換されたのち、１次元の畳み込み（Conv1D)演算が行われていることが分かります。   
"Spectrogram"は皆さんもうご存じですね。  
"Log-Mel"は、音の周波数が高くなればなるほどその差が分からなくなるという人間の聴覚特性を模倣するため、スペクトログラムに対しメルフィルタバンクと呼ばれる処理を行って、低周波は細かい粒度で、高周波は粗い粒度でデータを圧縮するものです。  
その後は、実は自然言語処理において機械翻訳を行う仕組みと概ね同じです。   
ですので、これは自然言語処理の回で詳しくお話ししましょう。  



<img src='https://raw.githubusercontent.com/openai/whisper/main/approach.png'>
Transformer Encoder blocksを通り特徴量化され、一方、右側では認識結果となるべき単語列（トークンの列）がTransformer Decode Blocksにより特徴量化され、その途中で音声特徴と対応付けられて（cross attention）、出力へとつながっています。

